# Step 2 — TOD Stop and Access Point Station Assignment

Reads the manually curated station, stop, and pedestrian access point datasets
from the project geodatabase, then spatially assigns each stop and access point
to its parent station using a progressive buffer analysis (150 ft → 300 ft → 500 ft).

Points that fall within multiple station buffers (conflicts) or outside all buffers
(no-match orphans) are exported as review layers to the shared GeoPackage for
manual resolution in GIS before the final re-integration step.

> **Pipeline order:** Run after `1_gtfs_tod_stop_classification.ipynb`.
> After running this notebook, open `tod_stops_review_v1` and
> `tod_access_review_v1` in GIS, resolve conflicts and orphans, then continue
> in `3_tod_station_assignment_reintegration.ipynb`.
> All data source paths are defined in `config.py`.


In [1]:
import pandas as pd
import geopandas as gpd
import uuid
from config import *

In [2]:
# All paths and layer names are imported from config.py
tod_db_gdb_path = TOD_DATABASE_GDB
tod_gtfs_db_gdb_path = TOD_DATABASE_GPKG

In [3]:
# Read stations and stops from curated geodatabase
stations_gdf = gpd.read_file(tod_db_gdb_path, layer=GDB_STATIONS_LAYER)
print(f"Loaded {len(stations_gdf)} stations from geodatabase")

stops_gdf = gpd.read_file(tod_db_gdb_path, layer=GDB_STOPS_LAYER)
print(f"Loaded {len(stops_gdf)} stops from geodatabase")

Loaded 593 stations from geodatabase
Loaded 11357 stops from geodatabase


## Per-agency pedestrian access point sources

Read each agency's access point file as a separate GeoDataFrame so schemas
can be inspected before building the merge.  Source paths are defined in
`config.py` under `ACCESS_PTS_SOURCES`.

In [4]:
# Read each agency's access point source as a separate GeoDataFrame
def _read_access_pts(path, layer=None):
    """Read a shapefile (zip) or GDB layer into a GeoDataFrame."""
    if path is None:
        return None
    kwargs = {"layer": layer} if layer is not None else {}
    return gpd.read_file(path, **kwargs)

access_pts_ba_gdf = _read_access_pts(ACCESS_PTS_BA_PATH, ACCESS_PTS_BA_LAYER)
access_pts_ct_gdf = _read_access_pts(ACCESS_PTS_CT_PATH, ACCESS_PTS_CT_LAYER)
access_pts_ac_gdf = _read_access_pts(ACCESS_PTS_AC_PATH, ACCESS_PTS_AC_LAYER)
access_pts_sc_gdf = _read_access_pts(ACCESS_PTS_SC_PATH, ACCESS_PTS_SC_LAYER)
access_pts_sf_gdf = _read_access_pts(ACCESS_PTS_SF_PATH, ACCESS_PTS_SF_LAYER)

_agency_gdfs = {
    "BA": access_pts_ba_gdf,
    "CT": access_pts_ct_gdf,
    "AC": access_pts_ac_gdf,
    "SC": access_pts_sc_gdf,
    "SF": access_pts_sf_gdf,
}

for agency, gdf in _agency_gdfs.items():
    if gdf is None:
        print(f"{agency}: not loaded (path is None)")
    else:
        print(f"{agency}: {len(gdf)} rows | CRS: {gdf.crs} | columns: {list(gdf.columns)}")

BA: 183 rows | CRS: EPSG:26911 | columns: ['stop_id', 'stop_name', 'stop_code', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'stop_url', 'tts_stop_n', 'platform_c', 'location_t', 'parent_sta', 'stop_timez', 'wheelchair', 'level_id', 'route_info', 'geometry']
CT: 260 rows | CRS: EPSG:4326 | columns: ['stop_id', 'stop_name', 'stop_code', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'stop_url', 'tts_stop_n', 'platform_c', 'location_t', 'parent_sta', 'stop_timez', 'wheelchair', 'level_id', 'route_info', 'geometry']
AC: 92 rows | CRS: EPSG:4326 | columns: ['access_id', 'station_id', 'acces_poin', 'location_t', 'stop_lat', 'stop_lon', 'stop_name', 'geometry']
SC: 239 rows | CRS: EPSG:4326 | columns: ['stop_id', 'stop_name', 'stop_code', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'location_t', 'parent_sta', 'level_id', 'route_info', 'vta_comm', 'geometry']
SF: 603 rows | CRS: EPSG:3857 | columns: ['stop_id_12', 'stop_name_', 'stop_code', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_i

In [5]:
# Inspect schema and sample row for each agency — run before building the merge
for agency, gdf in _agency_gdfs.items():
    if gdf is None:
        continue
    print(f"\n{'='*60}")
    print(f"Agency: {agency}  ({len(gdf)} rows)")
    print(f"CRS: {gdf.crs}")
    print(f"\nColumn dtypes:")
    print(gdf.dtypes.to_string())
    print(f"\nSample row:")
    print(gdf.iloc[0].to_string())


Agency: BA  (183 rows)
CRS: EPSG:26911

Column dtypes:
stop_id            str
stop_name          str
stop_code          str
stop_desc          str
stop_lat       float64
stop_lon       float64
zone_id            str
stop_url           str
tts_stop_n      object
platform_c      object
location_t         str
parent_sta         str
stop_timez         str
wheelchair         str
level_id           str
route_info         str
geometry      geometry

Sample row:
stop_id                                             COLS_5
stop_name                         Snell St Entrance / Exit
stop_code                                           COLS_5
stop_desc                          Pedestrian Access Point
stop_lat                                            37.754
stop_lon                                        -122.19679
zone_id                                           BA:79542
stop_url                                               NaN
tts_stop_n                                            None
platform_

In [8]:
# Normalize each agency GDF to a common schema before merging.
#
# Generic rename (applied to all agencies, only if target doesn't already exist):
#   stop_id     → access_id
#   stop_name   → access_point_name
#   location_t  → location_type
#   parent_sta  → station_id
#
# Per-agency pre-renames fix truncated/variant column names before the
# generic map runs (e.g. shapefile 10-char field name truncation).
#
# After renaming, keep only the five canonical columns.

RENAME_MAP = {
    "stop_id":    "access_id",
    "stop_name":  "access_point_name",
    "location_t": "location_type",
    "parent_sta": "station_id",
}

# Agency-specific renames applied BEFORE RENAME_MAP.
# Add entries here whenever a source file uses non-standard column names.
AGENCY_PRE_RENAME = {
    "SF": {
        "stop_id_12": "stop_id",
        "stop_name_": "stop_name",
    },
}

KEEP_COLS = ["access_id", "access_point_name", "location_type", "station_id", "geometry"]


def normalize_access_pts(gdf, agency):
    """Rename source columns to canonical names, then select KEEP_COLS."""
    if gdf is None:
        return None
    gdf = gdf.copy()

    # Apply any agency-specific pre-renames first
    pre_renames = AGENCY_PRE_RENAME.get(agency, {})
    for src, tgt in pre_renames.items():
        if src in gdf.columns:
            gdf = gdf.rename(columns={src: tgt})
            print(f"  {agency}: pre-rename '{src}' → '{tgt}'")

    # Apply generic rename map
    for src, tgt in RENAME_MAP.items():
        if src in gdf.columns and tgt not in gdf.columns:
            gdf = gdf.rename(columns={src: tgt})
        elif src in gdf.columns and tgt in gdf.columns:
            print(f"  {agency}: '{tgt}' already exists — skipping rename of '{src}'")

    missing = [c for c in KEEP_COLS if c != "geometry" and c not in gdf.columns]
    if missing:
        print(f"  {agency}: ⚠️  columns not found after rename: {missing}")

    cols = [c for c in KEEP_COLS if c in gdf.columns]
    return gdf[cols].copy()


access_pts_ba_norm = normalize_access_pts(access_pts_ba_gdf, "BA")
access_pts_ct_norm = normalize_access_pts(access_pts_ct_gdf, "CT")
access_pts_ac_norm = normalize_access_pts(access_pts_ac_gdf, "AC")
access_pts_sc_norm = normalize_access_pts(access_pts_sc_gdf, "SC")
access_pts_sf_norm = normalize_access_pts(access_pts_sf_gdf, "SF")

_norm_gdfs = {
    "BA": access_pts_ba_norm,
    "CT": access_pts_ct_norm,
    "AC": access_pts_ac_norm,
    "SC": access_pts_sc_norm,
    "SF": access_pts_sf_norm,
}

for agency, gdf in _norm_gdfs.items():
    if gdf is None:
        print(f"{agency}: not loaded")
    else:
        print(f"{agency}: {len(gdf)} rows | columns: {list(gdf.columns)}")

  SF: pre-rename 'stop_id_12' → 'stop_id'
  SF: pre-rename 'stop_name_' → 'stop_name'
BA: 183 rows | columns: ['access_id', 'access_point_name', 'location_type', 'station_id', 'geometry']
CT: 260 rows | columns: ['access_id', 'access_point_name', 'location_type', 'station_id', 'geometry']
AC: 92 rows | columns: ['access_id', 'access_point_name', 'location_type', 'station_id', 'geometry']
SC: 239 rows | columns: ['access_id', 'access_point_name', 'location_type', 'station_id', 'geometry']
SF: 603 rows | columns: ['access_id', 'access_point_name', 'location_type', 'station_id', 'geometry']


## Merge and clean merged access points

Concat all normalized agency GDFs into a single `access_pts_gdf`, then apply
deduplication and access_id gap-filling before spatial assignment.

In [14]:
# Concat all normalized agency GDFs into a single access_pts_gdf.
# Reproject each source to EPSG:4326 before concat to resolve CRS mismatches.
_to_concat = {k: v for k, v in _norm_gdfs.items() if v is not None}

for agency, gdf in _to_concat.items():
    print(f"  {agency}: {gdf.crs}")

access_pts_gdf = gpd.GeoDataFrame(
    pd.concat(
        [gdf.to_crs("EPSG:4326").assign(agency=agency) for agency, gdf in _to_concat.items()],
        ignore_index=True,
    ),
    crs="EPSG:4326",
)

print(f"\nMerged {len(_to_concat)} agency sources into {len(access_pts_gdf)} total access points")
for agency, gdf in _to_concat.items():
    print(f"  {agency}: {len(gdf)} rows")

  BA: EPSG:26911
  CT: EPSG:4326
  AC: EPSG:4326
  SC: EPSG:4326
  SF: EPSG:3857

Merged 5 agency sources into 1377 total access points
  BA: 183 rows
  CT: 260 rows
  AC: 92 rows
  SC: 239 rows
  SF: 603 rows


In [18]:
# Drop rows with duplicate access_id + geometry combinations.
# The agency column is retained for QA/QC purposes and will be dropped later.
before = len(access_pts_gdf)
access_pts_gdf = access_pts_gdf.drop_duplicates(subset=["access_id", "geometry"], keep="first")
print(f"Dropped {before - len(access_pts_gdf)} exact duplicate rows ({len(access_pts_gdf)} remaining)")

# access_id must be unique for downstream spatial assignment and joins.
# If the same access_id survived with two different geometries, reassign a new
# UUID to all but the first occurrence so uniqueness is guaranteed.
dup_id_mask = access_pts_gdf.duplicated(subset=["access_id"], keep="first")
n_dup_ids = dup_id_mask.sum()
if n_dup_ids > 0:
    print(f"⚠️  {n_dup_ids} rows share an access_id with a different geometry — reassigning new UUIDs")
    access_pts_gdf.loc[dup_id_mask, "access_id"] = [str(uuid.uuid4()) for _ in range(n_dup_ids)]
else:
    print("✅ access_id is unique after dedup")

assert access_pts_gdf["access_id"].duplicated().sum() == 0, "access_id is still not unique!"

Dropped 0 exact duplicate rows (1280 remaining)
⚠️  637 rows share an access_id with a different geometry — reassigning new UUIDs


In [19]:
# Assign UUID-based access_id to any rows missing one
null_id_mask = access_pts_gdf["access_id"].isna()
num_null = null_id_mask.sum()
print(f"Found {num_null} rows with null access_id")

if num_null > 0:
    access_pts_gdf.loc[null_id_mask, "access_id"] = [
        str(uuid.uuid4()) for _ in range(num_null)
    ]
    print(f"Assigned UUID access_id to {num_null} rows")
    print("Sample new access_ids:")
    print(access_pts_gdf.loc[null_id_mask, "access_id"].head(3).tolist())

# Confirm no remaining nulls or duplicates
dup_count = access_pts_gdf["access_id"].duplicated().sum()
print(f"\nRemaining null access_id:       {access_pts_gdf['access_id'].isna().sum()}")
print(f"Remaining duplicate access_id:  {dup_count}")
print(f"Total access points:            {len(access_pts_gdf)}")

Found 0 rows with null access_id

Remaining null access_id:       0
Remaining duplicate access_id:  0
Total access points:            1280


In [ ]:
# Join to the GTFS access_points layer written by Step 1 to assign station_id.
#
# station_id values from the per-agency source files are dropped before this
# join — they may be stale or use different IDs and cannot be trusted.
# The GTFS join is the sole authoritative source for initial station_id values;
# anything still unassigned after the join is handled by spatial assignment.

# Drop any station_id that came through from the per-agency source files
if "station_id" in access_pts_gdf.columns:
    n_source_assigned = access_pts_gdf["station_id"].notna().sum()
    print(f"Dropping {n_source_assigned} station_id value(s) from per-agency source data")
    access_pts_gdf = access_pts_gdf.drop(columns=["station_id"])

gtfs_access_pts_gdf = gpd.read_file(tod_gtfs_db_gdb_path, layer=GPKG_ACCESS_PTS_LAYER)
print(f"Loaded {len(gtfs_access_pts_gdf)} GTFS access points from GeoPackage (Step 1 output)")
print(f"  GTFS records with station_id: {gtfs_access_pts_gdf['station_id'].notna().sum()}")

access_pts_gdf = access_pts_gdf.merge(
    gtfs_access_pts_gdf[["access_id", "station_id"]],
    on="access_id",
    how="left",
)

n_assigned = access_pts_gdf["station_id"].notna().sum()
n_unassigned = access_pts_gdf["station_id"].isna().sum()
print(f"\nAfter GTFS station_id join:")
print(f"  Access points with station_id:    {n_assigned}")
print(f"  Access points without station_id: {n_unassigned} (will be resolved by spatial assignment)")

Before GTFS station_id join:
  Access points with station_id:    1225
  Access points without station_id: 55

Loaded 1236 GTFS access points from GeoPackage (Step 1 output)

After GTFS station_id join:
  Access points with station_id:    1225  (+0 from GTFS join)
  Access points without station_id: 55 (will be resolved by spatial assignment)


In [21]:
gtfs_access_pts_gdf

,access_id,access_point_name,station_id,location_type,geometry
0,mtc:powell_n1_concourse,NaN,mtc:powell,3,POINT (-122.40769 37.7842)
1,mtc:1586751243014,B1 Stairs (Concourse Level),mtc:powell,3,POINT (-122.40821 37.78379)
2,mtc:powell_n6_concourse_muni,Concourse,mtc:powell,3,POINT (-122.40629 37.78542)
3,mtc:powell_n8_concourse_elevator,Platform Elevator (Concourse Level),mtc:powell,3,POINT (-122.40625 37.78545)
4,mtc:1586757225033,BART Platform,mtc:powell,3,POINT (-122.4069 37.78489)
...,...,...,...,...,...
1231,SBR-02-NB,Elevator: San Bruno Caltrain Station Northbound,san_bruno,2,POINT (-122.41164 37.63094)
1232,SBR-01-SB,Elevator: San Bruno Caltrain Station Southbound,san_bruno,2,POINT (-122.41208 37.63101)
1233,SMT-01-NB,Elevator: San Mateo Caltrain Station Northbound,san_mateo,2,POINT (-122.3239 37.56825)
1234,SMT-02-SB,Elevator: San Mateo Caltrain Station Southbound,san_mateo,2,POINT (-122.3241 37.56812)


In [ ]:
access_pts_gdf

In [ ]:
# check if stops_gdf have any duplicate stop_id values
duplicate_stop_ids = stops_gdf[stops_gdf.duplicated(subset=["stop_id"], keep=False)]
print(f"Found {len(duplicate_stop_ids)} duplicate stop_id values in stops_gdf")
if len(duplicate_stop_ids) > 0:
    print("Sample duplicate stop_id values:")
    print(duplicate_stop_ids["stop_id"].unique()[:5].tolist())

Found 0 duplicate stop_id values in stops_gdf


In [ ]:
# Check current CRS and prepare for spatial assignment
print("Current CRS:")
print(f"Stations: {stations_gdf.crs}")
print(f"Stops: {stops_gdf.crs}")
print(f"Access Points: {access_pts_gdf.crs}")

# Define target CRS for accurate distance measurements
# Using UTM Zone 10N NAD83 - EPSG:26910 (meters)
target_crs = "EPSG:26910"

print(f"\nProjecting to {target_crs} for accurate distance measurements in meters...")

Current CRS:
Stations: EPSG:4326
Stops: EPSG:4326
Access Points: EPSG:4326

Projecting to EPSG:26910 for accurate distance measurements in meters...


In [ ]:
# Filter stops to only include TOD stops (tod_stop = 1) for station assignment
print("Filtering stops to only include TOD-applicable stops...")
print(f"Total stops before filtering: {len(stops_gdf)}")

# Check if tod_stop column exists and filter
if "tod_stop" in stops_gdf.columns:
    tod_stops_gdf = stops_gdf[stops_gdf["tod_stop"] == 1].copy()
    print(f"TOD stops (tod_stop=1): {len(tod_stops_gdf)}")
else:
    print("Warning: 'tod_stop' column not found. Using all stops.")
    tod_stops_gdf = stops_gdf.copy()

print(f"Proceeding with {len(tod_stops_gdf)} stops for station assignment")

Filtering stops to only include TOD-applicable stops...
Total stops before filtering: 11357
TOD stops (tod_stop=1): 755
Proceeding with 755 stops for station assignment


In [ ]:
# Project all datasets to target CRS for spatial operations
stations_proj = stations_gdf.to_crs(target_crs)
stops_proj = tod_stops_gdf.to_crs(target_crs)
access_pts_proj = access_pts_gdf.to_crs(target_crs)

print(f"Projected {len(stations_proj)} stations")
print(f"Projected {len(stops_proj)} TOD stops")
print(f"Projected {len(access_pts_proj)} access points")

Projected 593 stations
Projected 755 TOD stops
Projected 1280 access points


In [ ]:
def assign_station_ids_spatially(
    points_gdf, stations_gdf, id_col, station_id_col="station_id", max_distance=152.4
):
    """Assign station IDs to points using progressive buffer distances.

    Parameters:
    - points_gdf: GeoDataFrame of points to assign to stations
    - stations_gdf: GeoDataFrame of stations
    - id_col: column name for point IDs
    - station_id_col: column name for station IDs (existing or to create)
    - max_distance: maximum buffer distance in meters

    Returns:
    - Updated points_gdf with station assignments
    - DataFrame of conflicts (multiple station matches at the smallest buffer distance)
    - DataFrame of assignment statistics per buffer distance
    """

    # Create working copy
    points = points_gdf.copy()

    # Initialize station_id column if it doesn't exist
    if station_id_col not in points.columns:
        points[station_id_col] = None

    # Track assignment statistics and conflicts
    assignment_stats = []
    conflicts = []
    # Track IDs already flagged as conflicts so they are not re-processed
    already_conflicted = set()

    # Progressive buffer distances in meters (converted from feet)
    # 150ft ≈ 45.7m, 300ft ≈ 91.4m, 500ft ≈ 152.4m
    buffer_distances = [45.7, 91.4, 152.4]

    for buffer_dist in buffer_distances:
        feet_equiv = buffer_dist * 3.28084  # Convert meters to feet for display
        print(f"\n--- Buffer distance: {buffer_dist:.1f}m ({feet_equiv:.0f}ft) ---")

        # Only process points that are unassigned AND not yet flagged as conflicts
        unassigned_mask = points[station_id_col].isna() & ~points[id_col].isin(already_conflicted)
        unassigned_points = points[unassigned_mask].copy()

        if len(unassigned_points) == 0:
            print("All points assigned or in conflict!")
            break

        print(f"Processing {len(unassigned_points)} unassigned points")

        # Create buffers around stations
        station_buffers = stations_gdf.copy()
        station_buffers["geometry"] = stations_gdf.geometry.buffer(buffer_dist)

        # Spatial join to find intersections
        intersections = gpd.sjoin(
            unassigned_points[["geometry", id_col]],
            station_buffers[["geometry", "station_id"]],
            how="inner",
            predicate="intersects",
        )
        # Drop sjoin artifact column
        intersections = intersections.drop(columns=["index_right"], errors="ignore")

        if len(intersections) == 0:
            print(f"No assignments at {buffer_dist:.1f}m")
            continue

        # Find points with single vs multiple station matches
        match_counts = intersections.groupby(id_col).size()
        single_matches = match_counts[match_counts == 1].index
        multiple_matches = match_counts[match_counts > 1].index

        # Assign single matches
        single_match_data = intersections[intersections[id_col].isin(single_matches)]
        for idx, row in single_match_data.iterrows():
            point_mask = points[id_col] == row[id_col]
            points.loc[point_mask, station_id_col] = row["station_id"]

        # Flag conflicts and store for review — only at the buffer distance where they first appear
        if len(multiple_matches) > 0:
            conflict_data = intersections[intersections[id_col].isin(multiple_matches)].copy()
            # Add buffer distance info (first distance at which conflict was detected)
            conflict_data["buffer_distance"] = buffer_dist
            conflict_data["geometry"] = conflict_data.apply(
                lambda x: unassigned_points[unassigned_points[id_col] == x[id_col]].geometry.iloc[
                    0
                ],
                axis=1,
            )
            conflicts.append(conflict_data)
            # Prevent re-processing these IDs at larger buffer distances
            already_conflicted.update(multiple_matches.tolist())

        assigned_count = len(single_matches)
        conflict_count = len(multiple_matches)

        print(f"Assigned: {assigned_count}")
        print(f"New conflicts: {conflict_count}")

        assignment_stats.append(
            {
                "buffer_distance": buffer_dist,
                "assigned": assigned_count,
                "conflicts": conflict_count,
                "remaining_unassigned": unassigned_mask.sum() - assigned_count - conflict_count,
            }
        )

        if buffer_dist >= max_distance:
            break

    # Combine all conflicts
    all_conflicts = pd.concat(conflicts, ignore_index=True) if conflicts else pd.DataFrame()

    # Final statistics
    final_assigned = points[station_id_col].notna().sum()
    final_unassigned = points[station_id_col].isna().sum()

    print(f"\n=== FINAL RESULTS ===")
    print(f"Total points: {len(points)}")
    print(f"Assigned to stations: {final_assigned}")
    print(f"Unassigned (requires manual review): {final_unassigned}")
    print(f"Unique points with conflicts: {len(already_conflicted)}")
    print(f"Conflict rows (one per station candidate): {len(all_conflicts)}")

    return points, all_conflicts, pd.DataFrame(assignment_stats)

In [ ]:
# Apply spatial assignment to stops (only for stops without existing station_id)
print("=== ASSIGNING STATION IDs TO TOD STOPS ===")

stops_updated, stops_conflicts, stops_stats = assign_station_ids_spatially(
    stops_proj, stations_proj, id_col="stop_id", station_id_col="station_id", max_distance=152.4
)

print(f"\nTOD stops assignment statistics:")
print(stops_stats)

=== ASSIGNING STATION IDs TO TOD STOPS ===

--- Buffer distance: 45.7m (150ft) ---
Processing 505 unassigned points
Assigned: 457
New conflicts: 3

--- Buffer distance: 91.4m (300ft) ---
Processing 45 unassigned points
Assigned: 32
New conflicts: 2

--- Buffer distance: 152.4m (500ft) ---
Processing 11 unassigned points
Assigned: 5
New conflicts: 0

=== FINAL RESULTS ===
Total points: 755
Assigned to stations: 744
Unassigned (requires manual review): 11
Unique points with conflicts: 5
Conflict rows (one per station candidate): 10

TOD stops assignment statistics:
   buffer_distance  assigned  conflicts  remaining_unassigned
0             45.7       457          3                    45
1             91.4        32          2                    11
2            152.4         5          0                     6


In [ ]:
# Apply spatial assignment to access points
print("=== ASSIGNING STATION IDs TO ACCESS POINTS ===")

access_pts_updated, access_pts_conflicts, access_pts_stats = assign_station_ids_spatially(
    access_pts_proj,
    stations_proj,
    id_col="access_id",
    station_id_col="station_id",
    max_distance=152.4,
)

print(f"\nAccess points assignment statistics:")
print(access_pts_stats)

=== ASSIGNING STATION IDs TO ACCESS POINTS ===

--- Buffer distance: 45.7m (150ft) ---
Processing 1099 unassigned points
Assigned: 687
New conflicts: 13

--- Buffer distance: 91.4m (300ft) ---
Processing 399 unassigned points
Assigned: 232
New conflicts: 12

--- Buffer distance: 152.4m (500ft) ---
Processing 155 unassigned points
Assigned: 104
New conflicts: 16

=== FINAL RESULTS ===
Total points: 1280
Assigned to stations: 1204
Unassigned (requires manual review): 76
Unique points with conflicts: 41
Conflict rows (one per station candidate): 83

Access points assignment statistics:
   buffer_distance  assigned  conflicts  remaining_unassigned
0             45.7       687         13                   399
1             91.4       232         12                   155
2            152.4       104         16                    35


In [ ]:
# Handle conflicts and orphans, and prepare combined review outputs
print("=== HANDLING CONFLICTS AND PREPARING OUTPUTS ===")

# Convert back to original CRS for output
stops_final = stops_updated.to_crs(tod_stops_gdf.crs)
access_pts_final = access_pts_updated.to_crs(access_pts_gdf.crs)

# Review layer columns
STOP_REVIEW_COLS = [
    "stop_id",
    "station_id",
    "buffer_distance",
    "review_reason",
    "valid_station_assignment",
]
ACCESS_REVIEW_COLS = [
    "access_id",
    "station_id",
    "buffer_distance",
    "review_reason",
    "valid_station_assignment",
]


def make_review_gdf(rows, id_col, id_vals, station_ids, buffer_dists, reasons, geoms, crs):
    """Build a clean review GeoDataFrame from scratch with no sjoin metadata."""
    return gpd.GeoDataFrame(
        {
            id_col: id_vals,
            "station_id": station_ids,
            "buffer_distance": buffer_dists,
            "review_reason": reasons,
            "valid_station_assignment": [pd.NA] * len(id_vals),
        },
        geometry=gpd.GeoSeries(geoms, crs=crs),
        crs=crs,
    )


# --- Stops review layer ---
stop_review_rows = []

if len(stops_conflicts) > 0:
    conflicts_sub = stops_conflicts[["stop_id", "station_id", "buffer_distance", "geometry"]].copy()
    for _, row in conflicts_sub.iterrows():
        stop_review_rows.append(
            {
                "stop_id": row["stop_id"],
                "station_id": row["station_id"],
                "buffer_distance": row["buffer_distance"],
                "review_reason": "conflict",
                "valid_station_assignment": pd.NA,
                "geometry": row["geometry"],
            }
        )

conflicted_stop_ids = set(conflicts_sub["stop_id"].unique()) if len(stops_conflicts) > 0 else set()
orphan_stops = stops_final[
    stops_final["station_id"].isna() & ~stops_final["stop_id"].isin(conflicted_stop_ids)
]
for _, row in orphan_stops.iterrows():
    stop_review_rows.append(
        {
            "stop_id": row["stop_id"],
            "station_id": None,
            "buffer_distance": pd.NA,
            "review_reason": "no_match",
            "valid_station_assignment": pd.NA,
            "geometry": row["geometry"],
        }
    )

if stop_review_rows:
    stops_review_final = gpd.GeoDataFrame(
        stop_review_rows, geometry="geometry", crs=tod_stops_gdf.crs
    )
else:
    stops_review_final = gpd.GeoDataFrame(columns=STOP_REVIEW_COLS + ["geometry"])

n_conflict = sum(1 for r in stop_review_rows if r["review_reason"] == "conflict")
n_orphan = sum(1 for r in stop_review_rows if r["review_reason"] == "no_match")
print(f"\nStops requiring review: {len(stops_review_final)} rows")
print(f"  - Conflicts: {n_conflict}")
print(f"  - No match:  {n_orphan}")

# --- Access points review layer ---
access_review_rows = []

if len(access_pts_conflicts) > 0:
    conflicts_sub = access_pts_conflicts[
        ["access_id", "station_id", "buffer_distance", "geometry"]
    ].copy()
    for _, row in conflicts_sub.iterrows():
        access_review_rows.append(
            {
                "access_id": row["access_id"],
                "station_id": row["station_id"],
                "buffer_distance": row["buffer_distance"],
                "review_reason": "conflict",
                "valid_station_assignment": pd.NA,
                "geometry": row["geometry"],
            }
        )

conflicted_access_ids = (
    set(conflicts_sub["access_id"].unique()) if len(access_pts_conflicts) > 0 else set()
)
orphan_access = access_pts_final[
    access_pts_final["station_id"].isna()
    & ~access_pts_final["access_id"].isin(conflicted_access_ids)
]
for _, row in orphan_access.iterrows():
    access_review_rows.append(
        {
            "access_id": row["access_id"],
            "station_id": None,
            "buffer_distance": pd.NA,
            "review_reason": "no_match",
            "valid_station_assignment": pd.NA,
            "geometry": row["geometry"],
        }
    )

if access_review_rows:
    access_pts_review_final = gpd.GeoDataFrame(
        access_review_rows, geometry="geometry", crs=access_pts_gdf.crs
    )
else:
    access_pts_review_final = gpd.GeoDataFrame(columns=ACCESS_REVIEW_COLS + ["geometry"])

n_conflict = sum(1 for r in access_review_rows if r["review_reason"] == "conflict")
n_orphan = sum(1 for r in access_review_rows if r["review_reason"] == "no_match")
print(f"\nAccess points requiring review: {len(access_pts_review_final)} rows")
print(f"  - Conflicts: {n_conflict}")
print(f"  - No match:  {n_orphan}")

# Summary
print(f"\n=== FINAL SUMMARY ===")
print(
    f"TOD stops assigned to stations: {stops_final['station_id'].notna().sum()} / {len(stops_final)}"
)
print(
    f"Access points assigned to stations: {access_pts_final['station_id'].notna().sum()} / {len(access_pts_final)}"
)
print(f"Total rows requiring review: {len(stops_review_final) + len(access_pts_review_final)}")

=== HANDLING CONFLICTS AND PREPARING OUTPUTS ===

Stops requiring review: 16 rows
  - Conflicts: 10
  - No match:  6

Access points requiring review: 118 rows
  - Conflicts: 83
  - No match:  35

=== FINAL SUMMARY ===
TOD stops assigned to stations: 744 / 755
Access points assigned to stations: 1204 / 1280
Total rows requiring review: 134


In [ ]:
# Export results and review layers to GeoPackage
print(f"Exporting results to GeoPackage: {tod_gtfs_db_gdb_path}")

stops_final.to_file(tod_gtfs_db_gdb_path, layer=GPKG_TOD_STOPS_LAYER, driver="GPKG", mode="w")
print(f"✅ Exported {len(stops_final)} TOD stops")

access_pts_final.to_file(
    tod_gtfs_db_gdb_path, layer=GPKG_TOD_ACCESS_PTS_LAYER, driver="GPKG", mode="w"
)
print(f"✅ Exported {len(access_pts_final)} access points")

if len(stops_review_final) > 0:
    stops_review_final.to_file(
        tod_gtfs_db_gdb_path, layer=GPKG_TOD_STOPS_REVIEW_LAYER, driver="GPKG", mode="w"
    )
    print(f"✅ Exported {len(stops_review_final)} stop review rows")
    print(
        f"   - {(stops_review_final['review_reason'] == 'conflict').sum()} conflicts  |  {(stops_review_final['review_reason'] == 'no_match').sum()} no-match orphans"
    )

if len(access_pts_review_final) > 0:
    access_pts_review_final.to_file(
        tod_gtfs_db_gdb_path, layer=GPKG_TOD_ACCESS_REVIEW_LAYER, driver="GPKG", mode="w"
    )
    print(f"✅ Exported {len(access_pts_review_final)} access point review rows")
    print(
        f"   - {(access_pts_review_final['review_reason'] == 'conflict').sum()} conflicts  |  {(access_pts_review_final['review_reason'] == 'no_match').sum()} no-match orphans"
    )

print(f"\n🎉 Export complete!")
print(f"📁 Review outputs in: {tod_gtfs_db_gdb_path}")
print("\n📋 Next steps:")
print("1. In GIS, open tod_stops_review_v1 and tod_access_review_v1")
print("   - For each conflict row, set valid_station_assignment=1 on the correct station_id row")
print("   - For no-match orphans, fill in station_id and set valid_station_assignment=1")
print("2. Save and re-run the re-integration cells below")

Exporting results to GeoPackage: /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_database.gpkg
Removed stale layer: tod_stops
Removed stale layer: tod_access_points
Removed stale layer: tod_stops_review_v1
Removed stale layer: tod_access_review_v1
✅ Exported 755 TOD stops
✅ Exported 1280 access points
✅ Exported 16 stop review rows
   - 10 conflicts  |  6 no-match orphans
✅ Exported 118 access point review rows
   - 83 conflicts  |  35 no-match orphans

🎉 Export complete!
📁 Review outputs in: /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_database.gpkg

📋 Next steps:
1. In GIS, open tod_stops_review_v1 and tod_access_review_v1
   - For each conflict row, set valid_station_assignment=1 on the correct station_id row
   - For no-match orphans, fill in station_id and set valid_station_assignment=1
2. Save and re-run the r